# BL2403 — Phase 1: Preprocessing & EDA

This notebook covers:
1. Loading and inspecting all three cohorts (TCGA-BRCA, Sweden, METABRIC)
2. Filtering the STRING PPI network (`coexpression >= 190`)
3. Gene-set alignment across cohorts
4. Survival label binning (data-driven thresholds)
5. Expression and survival distribution visualisations
6. PCA-based batch-effect check

In [ ]:
import sys, os
# Point to repo root so src.* imports work inside the notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from sklearn.decomposition import PCA

from src.config import CANDIDATE_GENES, COEXPRESSION_THRESHOLD, PLOT_DIR
from src.preprocessing import (
    load_ppi, build_gene_index, build_edge_index,
    load_tcga, load_sweden, load_metabric,
    align_genes, compute_survival_thresholds, bin_survival,
    run_preprocessing,
)

## 1. PPI Network

In [ ]:
ppi = load_ppi()
print(f'PPI edges after filtering: {len(ppi):,}')
ppi.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ppi['coexpression'], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(COEXPRESSION_THRESHOLD, color='red', linestyle='--',
           label=f'Threshold = {COEXPRESSION_THRESHOLD}')
ax.set_xlabel('Coexpression Score')
ax.set_ylabel('Count')
ax.set_title('STRING PPI Coexpression Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Load mRNA-seq Cohorts

In [ ]:
tcga_mrna, tcga_clin = load_tcga()
print('TCGA mRNA (transposed):', tcga_mrna.shape)
tcga_mrna.iloc[:3, :5]

In [ ]:
sweden_mrna, sweden_clin = load_sweden()
print('Sweden mRNA (transposed):', sweden_mrna.shape)

metabric_mrna, metabric_clin = load_metabric()
print('METABRIC mRNA (transposed):', metabric_mrna.shape)

## 3. Full Preprocessing Pipeline

In [ ]:
preprocessed = run_preprocessing()

In [ ]:
tcga = preprocessed['tcga_df']
sweden = preprocessed['sweden_df']
metabric = preprocessed['metabric_df']

print('Common genes:', len(preprocessed['common_genes']))
print('Edge index shape:', preprocessed['edge_index'].shape)
print(f"TCGA samples: {len(tcga)}, Sweden: {len(sweden)}, METABRIC: {len(metabric)}")

## 4. Survival Distribution & Class Balance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, [('TCGA', tcga), ('Sweden', sweden), ('METABRIC', metabric)]):
    ax.hist(df['os_months'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(preprocessed['p33'], color='orange', linestyle='--', label=f'33rd={preprocessed["p33"]:.0f}m')
    ax.axvline(preprocessed['p66'], color='red', linestyle='--', label=f'66th={preprocessed["p66"]:.0f}m')
    ax.set_title(f'{name} – Overall Survival')
    ax.set_xlabel('Months')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.suptitle('Survival Distributions with Binning Thresholds', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
class_colors = ['steelblue', 'darkorange', 'green']
for ax, (name, df) in zip(axes, [('TCGA', tcga), ('Sweden', sweden), ('METABRIC', metabric)]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar([str(int(c)) for c in counts.index], counts.values, color=class_colors[:len(counts)])
    ax.set_title(f'{name} – Survival Class Balance')
    ax.set_xlabel('Class (0=Short, 1=Mid, 2=Long)')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 3, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Candidate Gene Expression

In [ ]:
common = preprocessed['common_genes']
candidate_present = [g for g in CANDIDATE_GENES if g in common]
print('Candidate genes in common set:', candidate_present)

if candidate_present:
    fig, axes = plt.subplots(1, len(candidate_present),
                             figsize=(4*len(candidate_present), 4))
    if len(candidate_present) == 1:
        axes = [axes]
    for ax, gene in zip(axes, candidate_present):
        for name, df, color in [('TCGA', tcga, 'steelblue'),
                                  ('Sweden', sweden, 'darkorange'),
                                  ('METABRIC', metabric, 'green')]:
            if gene in df.columns:
                ax.hist(df[gene].dropna(), bins=30, alpha=0.5, label=name, color=color)
        ax.set_title(gene)
        ax.set_xlabel('Expression (z-score)')
        ax.legend(fontsize=7)
    plt.suptitle('Candidate Gene Expression Across Cohorts', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('None of the candidate genes are in the common gene set (may need to check gene naming).')

## 6. PCA Batch-Effect Check

In [ ]:
all_expr = pd.concat([
    tcga.drop(columns=['os_months', 'os_status', 'label']),
    sweden.drop(columns=['os_months', 'os_status', 'label']),
    metabric.drop(columns=['os_months', 'os_status', 'label']),
], axis=0).fillna(0)

cohort_labels = (['TCGA'] * len(tcga) +
                 ['Sweden'] * len(sweden) +
                 ['METABRIC'] * len(metabric))

# Subsample for speed
if len(all_expr) > 2000:
    idx = np.random.choice(len(all_expr), 2000, replace=False)
    all_expr_sub = all_expr.iloc[idx]
    cohort_sub = [cohort_labels[i] for i in idx]
else:
    all_expr_sub = all_expr
    cohort_sub = cohort_labels

pca = PCA(n_components=2)
pc = pca.fit_transform(all_expr_sub.values)

fig, ax = plt.subplots(figsize=(7, 5))
colors_map = {'TCGA': 'steelblue', 'Sweden': 'darkorange', 'METABRIC': 'green'}
for cohort in set(cohort_sub):
    mask = np.array([c == cohort for c in cohort_sub])
    ax.scatter(pc[mask, 0], pc[mask, 1],
               label=cohort, alpha=0.4, s=8,
               color=colors_map.get(cohort, 'grey'))
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA – Batch Effect Check Across Cohorts')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
total_samples = len(tcga) + len(sweden) + len(metabric)
print('=== Dataset Summary ===')
print(f'Total samples: {total_samples}')
print(f'Common genes (nodes per graph): {len(common)}')
print(f'PPI edges (undirected): {preprocessed["edge_index"].size(1) // 2}')
print(f'Survival thresholds: Short<{preprocessed["p33"]:.1f}m, '
      f'Mid {preprocessed["p33"]:.1f}–{preprocessed["p66"]:.1f}m, '
      f'Long>{preprocessed["p66"]:.1f}m')
print()
all_labels = pd.concat([tcga['label'], sweden['label'], metabric['label']])
print('Global class distribution:')
print(all_labels.value_counts().sort_index())